# 1 class hidden

## Import libraries

In [1]:
import torch
from torch import nn, optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture
from collections import deque, defaultdict

# Import training, validation and test data

In [2]:
train = pd.read_csv('../train.csv')
val = pd.read_csv('../val.csv')
test = pd.read_csv('../test.csv')

'Infilteration' and 'DoS attacks-SlowHTTPTest' classes were aparently mislabeled, so we will exclude them, remaining 12 attack classes

In [3]:
classes_out = ['Infilteration','DoS attacks-SlowHTTPTest']
train = train.drop(index=train[train['Label'].isin(classes_out)].index)
val = val.drop(index=val[val['Label'].isin(classes_out)].index)
test = test.drop(index=test[test['Label'].isin(classes_out)].index)

11 attack classes will be joined into major attack classes
* FTP-BruteForce and SSH-Bruteforce -> BruteForce
* DoS attacks-GoldenEye, DoS attacks-Slowloris and DoS attacks-Hulk -> DoS
* DDoS attacks-LOIC-HTTP, DDOS attack-LOIC-UDP and DDOS attack-HOIC -> DDoS
* Brute Force -Web, Brute Force -XSS and SQL Injection -> DDoS

Resulting in 5 major attack classes: BruteForce, DoS, DDoS, Web and Bot, and Benign class 

In [4]:
train['Label'] = train['Label'].replace('FTP-BruteForce','BruteForce')
train['Label'] = train['Label'].replace('SSH-Bruteforce','BruteForce')
train['Label'] = train['Label'].replace('DoS attacks-GoldenEye','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Slowloris','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Hulk','DoS')
train['Label'] = train['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-HOIC','DDoS')
train['Label'] = train['Label'].replace('Brute Force -Web','Web')
train['Label'] = train['Label'].replace('Brute Force -XSS','Web')
train['Label'] = train['Label'].replace('SQL Injection','Web')


val['Label'] = val['Label'].replace('FTP-BruteForce','BruteForce')
val['Label'] = val['Label'].replace('SSH-Bruteforce','BruteForce')
val['Label'] = val['Label'].replace('DoS attacks-GoldenEye','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Slowloris','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Hulk','DoS')
val['Label'] = val['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-HOIC','DDoS')
val['Label'] = val['Label'].replace('Brute Force -Web','Web')
val['Label'] = val['Label'].replace('Brute Force -XSS','Web')
val['Label'] = val['Label'].replace('SQL Injection','Web')


test['Label'] = test['Label'].replace('FTP-BruteForce','BruteForce')
test['Label'] = test['Label'].replace('SSH-Bruteforce','BruteForce')
test['Label'] = test['Label'].replace('DoS attacks-GoldenEye','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Slowloris','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Hulk','DoS')
test['Label'] = test['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-HOIC','DDoS')
test['Label'] = test['Label'].replace('Brute Force -Web','Web')
test['Label'] = test['Label'].replace('Brute Force -XSS','Web')
test['Label'] = test['Label'].replace('SQL Injection','Web')

## Labels into numeric

In [5]:
labels_str = [i for i in train['Label'].value_counts().index]
print(labels_str)

for i, l in enumerate(labels_str):
    train['Label'] = train['Label'].replace(l, i)
    val['Label'] = val['Label'].replace(l, i)
    test['Label'] = test['Label'].replace(l, i)

['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']


C:\Users\Lincoln\AppData\Local\Temp\ipykernel_8000\2307520911.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['Label'] = train['Label'].replace(l, i)
C:\Users\Lincoln\AppData\Local\Temp\ipykernel_8000\2307520911.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['Label'] = val['Label'].replace(l, i)
C:\Users\Lincoln\AppData\Local\Temp\ipykernel_8000\2307520911.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `res

## Contrastive AutoEncoder Functions

<img src="..\img\contrastive_pairs.png" alt="Contrastive Pairs" width="60%"/>

In [6]:
def pick_pair(embeddings, labels, pick_embeddings, pick_labels):
    """
    Generates all pairs between embeddings (batch) and pick_embeddings (reference samples), with labels_pair indicating whether the pairs are positive (0) or negative (1). 
    """
    N, D = embeddings.shape
    M = pick_embeddings.shape[0]

    # Expands to all possible combinations
    A = embeddings.unsqueeze(0).expand(M, N, D).reshape(M * N, D)
    B = pick_embeddings.unsqueeze(1).expand(M, N, D).reshape(M * N, D)

    labels_exp = labels.unsqueeze(0).expand(M, N)
    pick_labels_exp = pick_labels.unsqueeze(1).expand(M, N)
    labels_pair = (labels_exp != pick_labels_exp).reshape(-1).float()

    return A, B, labels_pair

def ContrastiveLoss(input1, input2, y, margin=1.0):
    # y = 0 when input1 and input2 belong to the same class, and y = 1 otherwise.
    diff = input1 - input2
    dist_sq = torch.sum(torch.pow(diff,2),1)
    dist = torch.sqrt(dist_sq + 1e-10) # "Adds 1e-10 because if the sqrt is 0, the gradient becomes NaN.
    mdist = margin - dist
    mdist = F.relu(mdist)
    loss = ((1-y) * dist * dist) + (y * mdist * mdist)
    return torch.mean(loss)

def AEContrastive(input, output, embeddings, labels, pick_encoded, pick_labels, margin=1.0, lambda1=1.0):
    mseloss = nn.MSELoss()
    ae_loss = mseloss(input,output)

    A, B, labels_pair = pick_pair(embeddings, labels, pick_encoded, pick_labels)
    contrastive_loss = ContrastiveLoss(A, B, labels_pair, margin=margin)
    return ae_loss + contrastive_loss * lambda1, ae_loss, contrastive_loss

In [7]:
class contrastive_ae(nn.Module):
    def __init__(self, input_neurons, neurons):
        super().__init__()
        self.encoder0 = nn.Linear(input_neurons,64)
        self.encoder1 = nn.Linear(64,32)
        self.encoder2 = nn.Linear(32,16)
        self.encoder3 = nn.Linear(16,neurons)

        self.decoder0 = nn.Linear(neurons,16)
        self.decoder1 = nn.Linear(16,32)
        self.decoder2 = nn.Linear(32,64)
        self.decoder3 = nn.Linear(64,input_neurons)

        self.activation0 = nn.ReLU()
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X) # No activation function
        encoded = X
        X = self.activation0(self.decoder0(X))
        X = self.activation0(self.decoder1(X))
        X = self.activation0(self.decoder2(X))
        X = self.decoder3(X) # No activation function


        return X, encoded

In [8]:
class encoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder0 = list(model.children())[0]
        self.encoder1 = list(model.children())[1]
        self.encoder2 = list(model.children())[2]
        self.encoder3 = list(model.children())[3]
        self.activation0 = list(model.children())[8]
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        return X

In [9]:
from torch.utils.data import Dataset, DataLoader,TensorDataset
import random

# Prepare reference samples for each batch

def build_class_reference_batches(X_train, y_train, batch_size=512):
    classes = torch.unique(y_train).tolist()
    n_classes = len(classes)
    total_samples = len(X_train)
    dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    

    print(classes)

    # Maps each class to its indices
    class_to_indices = {
        int(cls.item()): (y_train == cls).nonzero(as_tuple=True)[0].tolist()
        for cls in torch.unique(y_train)
    }

    reference_list = []
    total_batches = len(train_loader)

    for batch_idx in range(total_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_samples)
        batch_indices = set(range(start, end))

        class_samples = []
        for cls in classes:
            candidates = [i for i in class_to_indices[cls] if i not in batch_indices]
            if not candidates:
                raise ValueError(f"Class {cls} has no samples outside of batch {batch_idx}")
            chosen = candidates[torch.randint(len(candidates), (1,)).item()]
            class_samples.append(chosen)

        reference_list.append(class_samples)


    return train_loader, reference_list

In [10]:
# Normalization function

def normalize(t):
    t_norm = (t - t.mean()) / t.std()
    return t_norm


In [12]:
array_hidden_classes = [[0],[2],[3],[4],[5]] # Class 1 correspond to benign class
filenames = ['0','2','3','4','5']

total_classes = len(labels_str)
for i in range(len(array_hidden_classes)):
    filename = f'{filenames[i]}_hidden'
    print(filename)
    hidden_classes = array_hidden_classes[i] # Classes hidden from training

    hidden_classes_index = list(train[train['Label'].isin(hidden_classes)].index)

    train_hidden = train.loc[hidden_classes_index]

    train_not_hidden = train.drop(hidden_classes_index)

    X_train = train_not_hidden.drop(columns=['Label'])
    y_train = train_not_hidden['Label']
    X_val = val.drop(columns=['Label'])
    y_val = val['Label'].values
    X_test = test.drop(columns=['Label'])
    y_test = test['Label'].values

    torch.manual_seed(123)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)
    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_loader, pick_list = build_class_reference_batches(X_train,y_train, batch_size=512)
    pick_samples = []
    pick_labels = []
    for l in pick_list:
        pick_samples.append(X_train[l])
        pick_labels.append(y_train[l])

    pick_samples = torch.stack(pick_samples)    
    pick_labels = torch.stack(pick_labels)


    neurons = total_classes - len(hidden_classes)
    print(f'{X_train.shape[1]} {neurons}')
    model = contrastive_ae(input_neurons=X_train.shape[1],neurons=neurons)
    model.to(device)

    learning_rate = 0.0001
    opt = optim.Adam(model.parameters(),lr=learning_rate)
    margin = 10 # Class separation parameter
    cae_lambda_1 = 1.0 # Contrastive learning weight 
    
    
    for epoch in range(250):
        running_loss = 0
        running_ael = 0
        running_cl = 0
        counter = 0
        for data in train_loader:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)
            ps = pick_samples[counter]
            pl = pick_labels[counter]
            ps = ps.to(device)
            pl = pl.to(device)
            counter += 1
            opt.zero_grad()
            outputs, encoded = model(inputs)
            _, pick_encoded = model(ps)
            tam_pe = pick_encoded.shape[0]
            enc_total = torch.cat((encoded,pick_encoded), dim=0)
            enc_total_norm = normalize(enc_total)
            encoded_norm = enc_total_norm[:-tam_pe]
            pick_encoded_norm = enc_total_norm[-tam_pe:]
            loss, ael, cl = AEContrastive(input=inputs, output=outputs, embeddings = encoded_norm, labels=labels, pick_encoded = pick_encoded_norm, pick_labels = pl, margin=margin, lambda1=cae_lambda_1)
            loss.backward()
            opt.step()
            running_loss += loss.item()
            running_ael += ael.item()
            running_cl += cl.item()
        print(f'filename: {filename} Epoch {epoch+1} loss:{running_loss/len(train_loader)} ael:{running_ael/len(train_loader)} cl:{running_cl/len(train_loader)}')

    model_e = encoder(model=model)
    model_e.to(device)

    model_e.eval()

    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_encoded = model_e(X_train.to(device))

    train_encoded = train_encoded.detach().cpu().numpy()

    train_encoded_df = pd.DataFrame(train_encoded)
    train_encoded_df['Label'] = y_train.detach().cpu().numpy().astype(int)
    display(train_encoded_df)

    X_val = torch.tensor(np.array(X_val), dtype=torch.float)

    val_encoded = model_e(X_val.to(device))

    val_encoded = val_encoded.detach().cpu().numpy()

    val_encoded_df = pd.DataFrame(val_encoded)

    val_encoded_df['Label'] = y_val

    display(val_encoded_df)

    X_test = torch.tensor(np.array(X_test), dtype=torch.float)

    test_encoded = model_e(X_test.to(device))

    test_encoded = test_encoded.detach().cpu().numpy()

    test_encoded_df = pd.DataFrame(test_encoded)

    test_encoded_df['Label'] = y_test

    display(test_encoded_df)

    train_encoded_df.to_csv(f'train_encoded_{filename}.csv',index=False)
    val_encoded_df.to_csv(f'val_encoded_{filename}.csv',index=False)
    test_encoded_df.to_csv(f'test_encoded_{filename}.csv',index=False)

0_hidden
cpu
[1.0, 2.0, 3.0, 4.0, 5.0]
82 5
filename: 0_hidden Epoch 1 loss:36.52888587171206 ael:0.028588073376522662 cl:36.500297817548244
filename: 0_hidden Epoch 2 loss:30.48726175871291 ael:0.017529986894408565 cl:30.469731782167322
filename: 0_hidden Epoch 3 loss:29.772329462568365 ael:0.017433226702677065 cl:29.754896247670192
filename: 0_hidden Epoch 4 loss:29.262142682238736 ael:0.0169204438526815 cl:29.245222233784954
filename: 0_hidden Epoch 5 loss:28.852165898151373 ael:0.016021811215044195 cl:28.83614410741455
filename: 0_hidden Epoch 6 loss:28.551514877122685 ael:0.015354266972703003 cl:28.53616062384001
filename: 0_hidden Epoch 7 loss:28.30608706449339 ael:0.014745508444405855 cl:28.291341557312627
filename: 0_hidden Epoch 8 loss:28.1163141494118 ael:0.01400754875171535 cl:28.10230661638792
filename: 0_hidden Epoch 9 loss:28.00462423578847 ael:0.013790540241884263 cl:27.990833682683604
filename: 0_hidden Epoch 10 loss:27.87828168511823 ael:0.013794924676391299 cl:27.8644

C:\Users\Lincoln\AppData\Local\Temp\ipykernel_8000\4243116071.py:86: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  X_train = torch.tensor(np.array(X_train), dtype=torch.float)
C:\Users\Lincoln\AppData\Local\Temp\ipykernel_8000\4243116071.py:87: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_train = torch.tensor(np.array(y_train), dtype=torch.float)


,0,1,2,3,4,Label
0,-0.053386,-0.053189,-0.042977,-0.079077,-0.078581,4
1,-0.058705,-0.051918,-0.074439,-0.072279,-0.040833,3
2,-0.077854,-0.070023,-0.068061,-0.068031,-0.067078,1
3,-0.054707,-0.067067,-0.066772,-0.047801,-0.062801,2
4,-0.078843,-0.070795,-0.068391,-0.068063,-0.067610,1
...,...,...,...,...,...,...
1270932,-0.058806,-0.052559,-0.073643,-0.071532,-0.041911,3
1270933,-0.053714,-0.069320,-0.065408,-0.043749,-0.066245,2
1270934,-0.058806,-0.052559,-0.073643,-0.071532,-0.041911,3
1270935,-0.085444,-0.073906,-0.067905,-0.070930,-0.072181,1


,0,1,2,3,4,Label
0,-0.053725,-0.069329,-0.065424,-0.043750,-0.066246,2
1,-0.057952,-0.051771,-0.074290,-0.071671,-0.040624,3
2,-0.053718,-0.069321,-0.065408,-0.043752,-0.066247,2
3,-0.082984,-0.076963,-0.064629,-0.064164,-0.077915,0
4,-0.082982,-0.076370,-0.064958,-0.064956,-0.077028,0
...,...,...,...,...,...,...
519951,-0.082985,-0.077052,-0.064580,-0.064046,-0.078047,0
519952,-0.082981,-0.075965,-0.065183,-0.065495,-0.076425,0
519953,-0.077936,-0.070029,-0.068020,-0.068109,-0.067109,1
519954,-0.079178,-0.070791,-0.067538,-0.068504,-0.068062,1


,0,1,2,3,4,Label
0,-0.053720,-0.069321,-0.065408,-0.043754,-0.066247,2
1,-0.078099,-0.070152,-0.067556,-0.068144,-0.067278,1
2,-0.053387,-0.053192,-0.042975,-0.079073,-0.078586,4
3,-0.058728,-0.051679,-0.074319,-0.072646,-0.040469,3
4,-0.079595,-0.071408,-0.068492,-0.067989,-0.068639,0
...,...,...,...,...,...,...
649942,-0.054371,-0.052876,-0.041912,-0.081598,-0.079855,4
649943,-0.054752,-0.054570,-0.041024,-0.079506,-0.082074,4
649944,-0.078664,-0.070900,-0.068478,-0.067734,-0.067688,1
649945,-0.078411,-0.070233,-0.067749,-0.068367,-0.067266,1


...

,0,1,2,3,4,Label
0,0.020301,0.048565,0.022761,0.021941,0.036099,0
1,0.045506,0.027528,0.062851,0.009310,-0.004456,4
2,-0.000567,0.010168,0.027457,0.065800,0.012319,3
3,0.023616,0.044950,0.022486,0.023244,0.036275,0
4,0.052435,0.017440,0.029375,0.039178,0.047848,1
...,...,...,...,...,...,...
1750627,0.000420,0.010403,0.027593,0.064326,0.012533,3
1750628,0.019939,0.047509,0.021574,0.021271,0.037057,0
1750629,0.048183,0.006340,0.028953,0.022703,0.037529,1
1750630,0.051754,0.024185,0.029759,0.044047,0.048044,1


,0,1,2,3,4,Label
0,0.025362,-0.010427,0.021430,0.014743,0.032444,2
1,-0.002271,0.010203,0.027721,0.064656,0.010508,3
2,0.025439,-0.010270,0.021344,0.015193,0.032742,2
3,0.020342,0.048539,0.022767,0.022007,0.036130,0
4,0.020755,0.048273,0.022831,0.022671,0.036443,0
...,...,...,...,...,...,...
519951,0.020281,0.048578,0.022758,0.021908,0.036083,0
519952,0.021035,0.048093,0.022874,0.023123,0.036657,0
519953,0.053127,0.017152,0.029682,0.038600,0.047774,1
519954,0.053113,0.018460,0.029799,0.040375,0.049625,1


,0,1,2,3,4,Label
0,0.025427,-0.010292,0.021380,0.015080,0.032653,2
1,0.052212,0.018176,0.028776,0.037790,0.046763,1
2,0.045463,0.027508,0.062818,0.009234,-0.004470,4
3,0.001226,0.009195,0.027030,0.064999,0.012225,3
4,0.023387,0.045060,0.022506,0.022906,0.036050,0
...,...,...,...,...,...,...
649942,0.039150,0.027490,0.066858,-0.000318,-0.013639,4
649943,0.033525,0.020132,0.060678,-0.011256,-0.013459,4
649944,0.052261,0.016182,0.029181,0.036825,0.048369,1
649945,0.052181,0.018067,0.028733,0.038030,0.046879,1


3_hidden
cpu
[0.0, 1.0, 2.0, 4.0, 5.0]
82 5
filename: 3_hidden Epoch 1 loss:37.35692320409711 ael:0.023719760050552118 cl:37.33320345797511
filename: 3_hidden Epoch 2 loss:30.486987739012108 ael:0.01583741099584317 cl:30.471150334711556
filename: 3_hidden Epoch 3 loss:29.242343649480016 ael:0.014931708561832405 cl:29.22741194332347
filename: 3_hidden Epoch 4 loss:28.45982389872806 ael:0.014926691877634655 cl:28.44489721750728
filename: 3_hidden Epoch 5 loss:27.97543596895684 ael:0.015039717555793839 cl:27.960396261776197
filename: 3_hidden Epoch 6 loss:27.713604138227833 ael:0.014897829325153596 cl:27.698706303860504
filename: 3_hidden Epoch 7 loss:27.507415047787276 ael:0.014772989281385946 cl:27.49264205941125
filename: 3_hidden Epoch 8 loss:27.309465023877728 ael:0.014573546917602764 cl:27.29489148503929
filename: 3_hidden Epoch 9 loss:27.127288985857234 ael:0.014426239902230983 cl:27.112862741977086
filename: 3_hidden Epoch 10 loss:26.953679816185147 ael:0.014338118093234186 cl:26.

KeyboardInterrupt: 